In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pmdarima as pm
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
df = pd.read_excel("../DataHackathon.xlsx")

df = df[df['Cantidad'] != 0]
print((df['Cantidad'] == 0).sum())  

mask_unidades = df['Unidad de venta'].isin(['KG', 'L'])
mask_tres_ceros = df['Cantidad'] % 1000 == 0
mask_conversion = mask_unidades & mask_tres_ceros
df.loc[mask_conversion, 'Cantidad'] = df.loc[mask_conversion, 'Cantidad'] / 1000

In [ ]:
articulo_objetivo = 'IVP11080'
df_articulo = df[df['Articulo'] == articulo_objetivo]

In [ ]:
from datetime import timedelta

# Agrupar por semana como antes
serie_limited = df_articulo.groupby(pd.Grouper(key='Creacion Orden de Venta', freq='W'))['Cantidad'].sum()
serie_limited = serie_limited.asfreq('W', fill_value=0)

# Obtener fecha mínima y calcular fecha de corte (2 años y 6 meses después)
fecha_inicio = serie_limited.index.min()
fecha_corte = fecha_inicio + pd.DateOffset(years=2, months=9)

# Filtrar la serie
serie_limited = serie_limited[serie_limited.index <= fecha_corte]

# Graficar
plt.figure(figsize=(12,6))
plt.plot(serie_limited, marker='o')
plt.title(f'Serie semanal limitada (2.5 años) - Artículo {articulo_objetivo}')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# SERIE COMPLETA (3 años)
serie = df_articulo.groupby(pd.Grouper(key='Creacion Orden de Venta', freq='W'))['Cantidad'].sum()
serie = serie.asfreq('W', fill_value=0)

# Graficar la serie completa
plt.figure(figsize=(12,6))
plt.plot(serie, marker='o')
plt.title(f'Serie semanal completa - Artículo {articulo_objetivo}')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Paso 5: Prueba de estacionariedad - ADF (Augmented Dickey-Fuller)
resultado_adf = adfuller(serie_limited)

adf_output = {
    'ADF Statistic': resultado_adf[0],
    'p-value': resultado_adf[1],
    'Lags Used': resultado_adf[2],
    'Number of Observations Used': resultado_adf[3],
    'Critical Values': resultado_adf[4]
}

adf_output

In [ ]:
# Creamos una figura con 2 subplots para ACF y PACF
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ACF: para definir q
plot_acf(serie.dropna(), lags=50, ax=axes[0])
axes[0].set_title('ACF - Función de Autocorrelación')

# PACF: para definir p
plot_pacf(serie.dropna(), lags=50, ax=axes[1], method='ywm')
axes[1].set_title('PACF - Función de Autocorrelación Parcial')

plt.show()

In [ ]:
from pmdarima import auto_arima

# ===============================
# MODELO AUTO_ARIMA CON ESTACIONALIDAD
# ===============================
modelo_auto = auto_arima(
    serie,                        # Tu serie temporal ya agregada (frecuencia semanal)
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=1,                          # Diferenciación regular (ADF indicó no estacionaria)
    start_P=0, start_Q=0,
    max_P=2, max_Q=2,
    D=1,                          # Diferenciación estacional
    seasonal=True,               # Habilitar estacionalidad
    m=52,                        # Frecuencia estacional: 52 semanas = 1 año
    stepwise=True,               # Algoritmo rápido y eficiente
    trace=True,                  # Mostrar progreso
    error_action='ignore',       # Ignorar errores
    suppress_warnings=True,      # Silenciar warnings
                       # Limitar cantidad de modelos a probar (opcional)
    information_criterion='aic'  # Criterio para selección del mejor modelo
)

# Mostrar resumen del mejor modelo encontrado
print(modelo_auto.summary())

# Guardar el orden del modelo seleccionado
best_order = modelo_auto.order
best_seasonal_order = modelo_auto.seasonal_order

print(f"Mejor ARIMA: {best_order}")
print(f"Mejor SARIMA: {best_seasonal_order}")

***PREDICCION SEIS MES DENTRO DE LOS DATOS***

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Entrena el modelo con los mejores parámetros
final_model = SARIMAX(serie_limited,
                      order=(1, 1, 1),
                      seasonal_order=(2, 1, 1, 52),
                      enforce_stationarity=False,
                      enforce_invertibility=False)

final_result = final_model.fit(disp=False)

# Resumen del modelo entrenado
print(final_result.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import shapiro

# 1. Extraer los residuos del modelo
residuals = final_result.resid

# 2. Gráfica de residuos en el tiempo
plt.figure(figsize=(12,4))
plt.plot(residuals)
plt.title('Residuos del modelo')
plt.show()

# 3. Histograma de los residuos
plt.figure(figsize=(8,4))
sns.histplot(residuals, kde=True, bins=30)
plt.title('Histograma de los residuos')
plt.show()

# 4. Gráfica Q-Q para ver normalidad
import statsmodels.api as sm
sm.qqplot(residuals, line='s')
plt.title('Q-Q plot de los residuos')
plt.show()

# 5. Autocorrelación de residuos
fig, ax = plt.subplots(1,2, figsize=(16,4))
plot_acf(residuals, ax=ax[0])
plot_pacf(residuals, ax=ax[1])
plt.show()

# 6. Ljung-Box test
lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print("Ljung-Box test (lag=10):")
print(lb_test)

# 7. Shapiro-Wilk test para normalidad
shapiro_test = shapiro(residuals)
print(f"\nShapiro-Wilk Test:")
print(f"  - Estadístico: {shapiro_test[0]:.4f}")
print(f"  - p-value: {shapiro_test[1]:.4f}")

In [ ]:
# Número de pasos a pronosticar (12 semanas = ~3 meses)
steps_forecast = 12

# Predicción a futuro
pred_uc = final_result.get_forecast(steps=steps_forecast)

# Media e intervalos
forecast_mean = pred_uc.predicted_mean
forecast_ci = pred_uc.conf_int()

forecast_mean_adjusted = forecast_mean.clip(lower=0)

In [ ]:
# Asegúrate de tener estas variables listas:
# - serie: la serie completa (3 años)
# - forecast_mean: las predicciones de 3 meses
# - forecast_ci: intervalos de confianza
# - final_result: modelo entrenado con serie_limited (2.5 años)

# Graficar la serie completa con la predicción sobrepuesta
plt.figure(figsize=(12,6))

# Serie histórica completa
plt.plot(serie, label='Serie histórica (3 años)', color='blue')

# Forecast de 3 meses
plt.plot(forecast_mean_adjusted, label='Pronóstico (6 meses)', color='green')

# Banda de intervalo de confianza
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='lightgreen', alpha=0.5, label='Intervalo de confianza')

# Formato del gráfico
plt.title(f'Serie completa con Pronóstico de 3 meses - Artículo {articulo_objetivo}')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Repetimos la predicción para definir forecast_mean
pasos = 12  # 25 semanas = 6 meses

pred_3meses = final_result.get_forecast(steps=pasos)
forecast_mean = pred_3meses.predicted_mean

# Sumar la cantidad total estimada para 3 meses
total_pronosticado = forecast_mean.sum()
print(f'La cantidad de inventario de la variable {articulo_objetivo} para los siguientes 3 meses es {total_pronosticado.round()}')

In [ ]:
# Obtener la fecha final del entrenamiento (última fecha de serie_limited)
fecha_corte = serie_limited.index.max()

# Filtrar la serie real desde el corte hasta 25 semanas después
serie_real = serie[(serie.index > fecha_corte) & (serie.index <= fecha_corte + pd.DateOffset(weeks=12))]

# Sumar la cantidad real observada
total_real = serie_real.sum()

# Mostrar
print(f'Cantidad real observada en los siguientes 3 meses: {total_real}')

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Intersección de índices para evitar KeyError
common_index = forecast_mean.index.intersection(serie.index)

# Extraer valores reales y predichos
y_true = serie[common_index]
y_pred = forecast_mean[common_index]

# Calcular errores
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true.replace(0, np.nan))) * 100

print("Errores de predicción dentro del rango:")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_true, label="Real", marker='o')
plt.plot(y_pred, label="Predicción", marker='x')
plt.title("Comparación: Valores Reales vs Predicción ARIMA (3 meses)")
plt.xlabel("Fecha")
plt.ylabel("Cantidad")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

***PREDICCION PRIMER MES DESPUES DE LOS DATOS***

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Entrena el modelo con los mejores parámetros
final_model = SARIMAX(serie,
                      order=(1, 1, 1),
                      seasonal_order=(2, 1, 1, 52),
                      enforce_stationarity=False,
                      enforce_invertibility=False)

final_result = final_model.fit(disp=False)

# Resumen del modelo entrenado
print(final_result.summary())

In [ ]:
# Número de pasos a pronosticar (4 semanas = ~1 meses)
steps_forecast = 4

# Predicción a futuro
pred_uc = final_result.get_forecast(steps=steps_forecast)

# Media e intervalos
forecast_mean = pred_uc.predicted_mean
forecast_ci = pred_uc.conf_int()
forecast_mean_adjusted = forecast_mean.clip(lower=0)

# Crear gráfico
plt.figure(figsize=(12,6))
plt.plot(serie, label='Serie histórica', color='blue')
plt.plot(forecast_mean_adjusted, label='Pronóstico (3 meses)', color='green')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='lightgreen', alpha=0.5, label='Intervalo de confianza')
plt.title('Pronóstico de ventas para el artículo IVP04039 (1 meses)')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Repetimos la predicción para definir forecast_mean
pasos = 4  # 4 semanas = 1 meses

pred_1meses = final_result.get_forecast(steps=pasos)
forecast_mean = pred_1meses.predicted_mean

# Sumar la cantidad total estimada para 3 meses
total_pronosticado_1 = forecast_mean.sum()
print(f'La cantidad de inventario de la variable {articulo_objetivo} para el siguiente mes es {total_pronosticado_1.round()}')

***PREDICCION TRES MESES DESPUES DE LOS DATOS***

In [ ]:
# Número de pasos a pronosticar (12 semanas = ~3 meses)
steps_forecast = 12

# Predicción a futuro
pred_uc = final_result.get_forecast(steps=steps_forecast)

# Media e intervalos
forecast_mean = pred_uc.predicted_mean
forecast_ci = pred_uc.conf_int()
forecast_mean_adjusted = forecast_mean.clip(lower=0)

# Crear gráfico
plt.figure(figsize=(12,6))
plt.plot(serie, label='Serie histórica', color='blue')
plt.plot(forecast_mean_adjusted, label='Pronóstico (3 meses)', color='green')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='lightgreen', alpha=0.5, label='Intervalo de confianza')
plt.title('Pronóstico de ventas para el artículo IVP04039 (3 meses)')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Repetimos la predicción para definir forecast_mean
pasos = 12  # 12 semanas = 3 meses

pred_3meses = final_result.get_forecast(steps=pasos)
forecast_mean = pred_3meses.predicted_mean

# Sumar la cantidad total estimada para 3 meses
total_pronosticado_3 = forecast_mean.sum()
print(f'La cantidad de inventario de la variable {articulo_objetivo} para los siguientes 3 meses es {total_pronosticado_3.round()}')

***PREDICCION SEIS MESES DESPUES DE LOS DATOS***

In [ ]:
# Número de pasos a pronosticar (26 semanas = ~6 meses)
steps_forecast = 26

# Predicción a futuro
pred_uc = final_result.get_forecast(steps=steps_forecast)

# Media e intervalos
forecast_mean = pred_uc.predicted_mean
forecast_ci = pred_uc.conf_int()
forecast_mean_adjusted = forecast_mean.clip(lower=0)

# Crear gráfico
plt.figure(figsize=(12,6))
plt.plot(serie, label='Serie histórica', color='blue')
plt.plot(forecast_mean_adjusted, label='Pronóstico (3 meses)', color='green')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='lightgreen', alpha=0.5, label='Intervalo de confianza')
plt.title('Pronóstico de ventas para el artículo IVP04039 (6 meses)')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Repetimos la predicción para definir forecast_mean
pasos = 26  # 26 semanas = 6 meses

pred_6meses = final_result.get_forecast(steps=pasos)
forecast_mean = pred_6meses.predicted_mean

# Sumar la cantidad total estimada para 3 meses
total_pronosticado_6 = forecast_mean.sum()
print(f'La cantidad de inventario de la variable {articulo_objetivo} para los siguientes 6 meses es {total_pronosticado_6.round()}')

***PREDICCION DOCE MESES DESPUES DE LOS DATOS***

In [ ]:
# Número de pasos a pronosticar (12 semanas = ~3 meses)
steps_forecast = 52

# Predicción a futuro
pred_uc = final_result.get_forecast(steps=steps_forecast)

# Media e intervalos
forecast_mean = pred_uc.predicted_mean
forecast_ci = pred_uc.conf_int()
forecast_mean_adjusted = forecast_mean.clip(lower=0)

# Crear gráfico
plt.figure(figsize=(12,6))
plt.plot(serie, label='Serie histórica', color='blue')
plt.plot(forecast_mean_adjusted, label='Pronóstico (12 meses)', color='green')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='lightgreen', alpha=0.5, label='Intervalo de confianza')
plt.title(f'Pronóstico de ventas para el artículo {articulo_objetivo} (12 meses)')
plt.xlabel('Fecha')
plt.ylabel('Cantidad vendida')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Repetimos la predicción para definir forecast_mean
pasos = 52  # 26 semanas = 6 meses

pred_12meses = final_result.get_forecast(steps=pasos)
forecast_mean = pred_12meses.predicted_mean

# Sumar la cantidad total estimada para 3 meses
total_pronosticado_12 = forecast_mean.sum()
print(f'La cantidad de inventario de la variable {articulo_objetivo} para los siguientes 12 meses es {total_pronosticado_12.round()}')

In [ ]:
# Leer el Excel existente
df_existente = pd.read_excel("PronosticoArticulosSARIMA.xlsx")

# Supón que tienes esta nueva fila lista en df_nueva_fila
# (puedes repetir esto en varios notebooks)
df_nueva_fila = pd.DataFrame([[articulo_objetivo, total_pronosticado_1, total_pronosticado_3,
                                total_pronosticado_6, total_pronosticado_12, df[df['Articulo'] == articulo_objetivo]['Unidad de venta'].iloc[0]]],
                             columns=['Articulo', 'Pronostico Mes 1', 'Pronostico Mes 3',
                                      'Pronostico Mes 6', 'Pronostico Mes 12', 'Unidad de Venta'])

# Concatenar
df_actualizado = pd.concat([df_existente, df_nueva_fila], ignore_index=True)

# Guardar sobrescribiendo el archivo
df_actualizado.to_excel("PronosticoArticulosSARIMA.xlsx", index=False)